In [1]:
import os
import numpy as np
import pandas as pd

In [2]:
from google.colab import drive
drive.mount("/content/drive/")

Drive already mounted at /content/drive/; to attempt to forcibly remount, call drive.mount("/content/drive/", force_remount=True).


In [3]:
DRIVE_PROJECT_PATH = "/content/drive/MyDrive/NLP_Projects/01_HateSpeech/01_R1/10_TwoSetsLabels"
RAW_DATA_PATH1 = os.path.join(DRIVE_PROJECT_PATH, "hatexplain_model_ready_with_ai_labels.csv")
RAW_DATA_PATH2 = os.path.join(DRIVE_PROJECT_PATH, "hatexplain_human_plus_mini.csv")

print("File 1:", RAW_DATA_PATH1)
print("File 2:", RAW_DATA_PATH2)

File 1: /content/drive/MyDrive/NLP_Projects/01_HateSpeech/01_R1/10_TwoSetsLabels/hatexplain_model_ready_with_ai_labels.csv
File 2: /content/drive/MyDrive/NLP_Projects/01_HateSpeech/01_R1/10_TwoSetsLabels/hatexplain_human_plus_mini.csv


In [4]:
df_raw1 = pd.read_csv(RAW_DATA_PATH1, index_col=0)
df_raw2 = pd.read_csv(RAW_DATA_PATH2, index_col=0)

print(f"Loaded df_raw1: {df_raw1.shape}")
print(f"Loaded df_raw2: {df_raw2.shape}")

Loaded df_raw1: (20148, 12)
Loaded df_raw2: (20148, 27)


In [5]:
print("df_raw1:", df_raw1.shape)
print("df_raw2:", df_raw2.shape)

y1 = df_raw1["ai_label_mini"]
y2 = df_raw2["ai_label_mini"]

print("len(y1):", len(y1))
print("len(y2):", len(y2))

mask = y1.notna() & y2.notna()
print("len(mask):", len(mask), "mask TRUE:", mask.sum())

df_raw1: (20148, 12)
df_raw2: (20148, 27)
len(y1): 20148
len(y2): 20148
len(mask): 40296 mask TRUE: 0


In [6]:
# ---- Sanity: same number of rows ----
assert df_raw1.shape[0] == df_raw2.shape[0], "Row counts differ—cannot do row-wise agreement."

# ---- Sanity: row alignment (text must match row-by-row) ----
TEXT_COL = "text"
assert TEXT_COL in df_raw1.columns and TEXT_COL in df_raw2.columns, "Missing text column."

same_text = (df_raw1[TEXT_COL].fillna("").astype(str).values
             == df_raw2[TEXT_COL].fillna("").astype(str).values)
print("Row-aligned text identical rate:", same_text.mean())

assert same_text.all(), "Text differs by row—files not aligned. Merge by key instead."

Row-aligned text identical rate: 1.0


In [7]:
import numpy as np
import pandas as pd

TEXT_COL = "text"
LABEL_COL = "ai_label_mini"
PROB_COLS = ["ai_p_normal_mini", "ai_p_offensive_mini", "ai_p_hate_mini"]

d1 = df_raw1[[TEXT_COL, LABEL_COL] + [c for c in PROB_COLS if c in df_raw1.columns]].copy()
d2 = df_raw2[[TEXT_COL, LABEL_COL]].copy()

# Important: ensure text is string for stable merge
d1[TEXT_COL] = d1[TEXT_COL].fillna("").astype(str)
d2[TEXT_COL] = d2[TEXT_COL].fillna("").astype(str)

# If duplicates exist, keep first occurrence (or you can aggregate; see note below)
d1 = d1.drop_duplicates(subset=[TEXT_COL]).reset_index(drop=True)
d2 = d2.drop_duplicates(subset=[TEXT_COL]).reset_index(drop=True)

m = d1.merge(d2, on=TEXT_COL, how="inner", suffixes=("_run1", "_run2"))
print("Merged shape:", m.shape)

# Hard-label agreement
mask = m[f"{LABEL_COL}_run1"].notna() & m[f"{LABEL_COL}_run2"].notna()
agree_rate = (m.loc[mask, f"{LABEL_COL}_run1"].values == m.loc[mask, f"{LABEL_COL}_run2"].values).mean()
mismatch = (m.loc[mask, f"{LABEL_COL}_run1"].values != m.loc[mask, f"{LABEL_COL}_run2"].values).sum()

print("Compared rows:", int(mask.sum()))
print(f"Agreement: {agree_rate:.4f}")
print("Mismatches:", int(mismatch))

# Confusion matrix
display(pd.crosstab(m.loc[mask, f"{LABEL_COL}_run1"], m.loc[mask, f"{LABEL_COL}_run2"]))

Merged shape: (20109, 6)
Compared rows: 20109
Agreement: 0.9298
Mismatches: 1412


ai_label_mini_run2,HATE,NORMAL,OFFENSIVE
ai_label_mini_run1,,,
HATE,10273,14,150
NORMAL,40,2937,96
OFFENSIVE,664,448,5487


In [8]:
if all(c in m.columns for c in [f"{c}" for c in PROB_COLS]):
    P = m.loc[mask, PROB_COLS].to_numpy(dtype=np.float64)
    P = P / np.clip(P.sum(axis=1, keepdims=True), 1e-12, None)
    ent = -(P * np.log(np.clip(P, 1e-12, 1.0))).sum(axis=1)

    agree_vec = (m.loc[mask, f"{LABEL_COL}_run1"].values == m.loc[mask, f"{LABEL_COL}_run2"].values)

    ent_bins = pd.qcut(ent, 5, labels=False, duplicates="drop")
    tmp = pd.DataFrame({"entropy_bin": ent_bins, "agree": agree_vec})
    display(tmp.groupby("entropy_bin")["agree"].agg(["mean", "count"]))
else:
    print("Probability columns not present in merged dataframe; skipping entropy-binned analysis.")

,mean,count
entropy_bin,,
0,0.948447,16779
1,0.835736,3330


In [9]:
from sklearn.metrics import cohen_kappa_score

y1 = m.loc[mask, "ai_label_mini_run1"].astype(str).values
y2 = m.loc[mask, "ai_label_mini_run2"].astype(str).values

kappa = cohen_kappa_score(y1, y2)
print("Cohen's kappa:", kappa)

# Optional: ordinal-ish weighting (only if you justify an order normal<offensive<hate)
order = {"NORMAL":0, "OFFENSIVE":1, "HATE":2}
y1o = np.array([order[v] for v in y1])
y2o = np.array([order[v] for v in y2])
kappa_w = cohen_kappa_score(y1o, y2o, weights="quadratic")
print("Weighted kappa (quadratic):", kappa_w)

Cohen's kappa: 0.8824403845491238
Weighted kappa (quadratic): 0.9295302007595266
